# Proyecto Final - Agente RL para Connect-4
**Curso:** Fundamentos de Inteligencia Artificial - Universidad de La Sabana, 2026.1  
**Rama:** Martin-Jerez

## Arquitectura del agente

El `QLearningAgent` combina dos ideas del curso:

1. **Q-Learning offline (Diapo 12):** auto-juego con truco bipolar aprende pesos `w` para `V(s) = w * psi(s)`.
2. **Alpha-Beta Minimax depth=4:** durante el juego busca 4 niveles usando `V(s)` como heuristica de hoja — detecta forks y tacticas implicitamente.

**Flujo del notebook:** el agente se entrena **UNA SOLA VEZ** en la celda de Setup. Los experimentos siguientes reusan los pesos aprendidos sin reentrenar.

In [ ]:
import numpy as np
import random
import time
import matplotlib.pyplot as plt
from rl_agent import QLearningAgent
from mcts_random import MCTSAgentRandom

ROWS, COLS = 6, 7

def apply_move(board, col, player):
    nb = board.copy()
    for r in range(ROWS - 1, -1, -1):
        if nb[r, col] == 0:
            nb[r, col] = player; break
    return nb

def check_winner(board):
    for r in range(ROWS):
        for c in range(COLS):
            p = board[r, c]
            if p == 0: continue
            if c+3<COLS and all(board[r,c+i]==p for i in range(4)): return p
            if r+3<ROWS and all(board[r+i,c]==p for i in range(4)): return p
            if r+3<ROWS and c+3<COLS and all(board[r+i,c+i]==p for i in range(4)): return p
            if r+3<ROWS and c-3>=0  and all(board[r+i,c-i]==p for i in range(4)): return p
    return 0

def random_act(board):
    free = [c for c in range(COLS) if board[0, c] == 0]
    return random.choice(free) if free else 3

def play_game(agent_a, agent_b, a_is_minus1=True):
    board = np.zeros((ROWS, COLS), dtype=int)
    a_player = -1 if a_is_minus1 else 1
    current = -1
    while True:
        free = [c for c in range(COLS) if board[0, c] == 0]
        if not free: return 0
        act_fn = (agent_a if callable(agent_a) else agent_a.act) if current == a_player \
                 else (agent_b if callable(agent_b) else agent_b.act)
        col = act_fn(board)
        board = apply_move(board, col, current)
        w = check_winner(board)
        if w != 0: return 1 if w == a_player else -1
        current = -current

def run_series(agent, opponent, n_games=100):
    wins = draws = losses = 0
    for i in range(n_games):
        r = play_game(agent, opponent, a_is_minus1=(i % 2 == 0))
        if r == 1: wins += 1
        elif r == 0: draws += 1
        else: losses += 1
    return wins, draws, losses

def make_agent(weights, depth=4, player=-1):
    """Instancia un agente con pesos ya aprendidos, sin reentrenar."""
    a = QLearningAgent(player=player, n_episodes=1, search_depth=depth)
    a.weights = weights.copy()
    return a

print('Utilidades cargadas.')


## Setup: Entrenamiento unico

**Ejecutar solo una vez.** Los pesos quedan en `W_BASE` y se reusan en todos los experimentos.

In [ ]:
N_EPISODES = 5000
DEPTH = 4

print(f'Entrenando QLearningAgent con {N_EPISODES} episodios...')
t0 = time.time()
ql_base = QLearningAgent(player=-1, n_episodes=N_EPISODES, search_depth=DEPTH)
ql_base.mount()
t_train = time.time() - t0

W_BASE = ql_base.weights.copy()  # pesos compartidos en todo el notebook

mcts200 = MCTSAgentRandom(player=1, num_simulations=200)
mcts200.mount()

print(f'Entrenamiento completado en {t_train:.1f}s')
print(f'Pesos aprendidos W_BASE: {W_BASE.round(3)}')
print('Listo. Los experimentos siguientes reusan W_BASE sin reentrenar.')


## Experimento 1: Impacto de `n_episodes` vs. agente aleatorio

Para este experimento SI entrenamos agentes nuevos con distintos presupuestos de episodios — es precisamente lo que queremos medir. **El agente de 5000 ep reusa W_BASE directamente.**

In [ ]:
episode_values = [100, 500, 1000, 2000, 5000]
win_rates = []

for n_ep in episode_values:
    if n_ep == N_EPISODES:
        # Reusa W_BASE, sin reentrenar
        agent = make_agent(W_BASE, depth=DEPTH)
        print(f'n_episodes={n_ep} (reusando W_BASE)...', end=' ')
    else:
        agent = QLearningAgent(player=-1, n_episodes=n_ep, search_depth=DEPTH)
        agent.mount()
        print(f'n_episodes={n_ep}...', end=' ')
    wins, draws, losses = run_series(agent, random_act, n_games=100)
    win_rates.append(wins)
    print(f'W={wins} D={draws} L={losses}')

plt.figure(figsize=(8, 4))
plt.plot(episode_values, win_rates, marker='o', linewidth=2)
plt.axhline(50, color='red', linestyle='--', label='Umbral 50%')
plt.xlabel('n_episodes'); plt.ylabel('Victorias / 100 partidas')
plt.title('QLearningAgent vs. Agente Aleatorio')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()


## Experimento 2: Agente fuerte (5000 ep) vs. agente debil (500 ep)

El agente fuerte reusa W_BASE. Solo se entrena el agente debil.

In [ ]:
# Agente fuerte: reusa W_BASE, sin reentrenar
strong = make_agent(W_BASE, depth=DEPTH, player=-1)

# Agente debil: unico entrenamiento de 500 ep
print('Entrenando agente debil con 500 episodios...')
weak = QLearningAgent(player=1, n_episodes=500, search_depth=DEPTH)
weak.mount()

wins, draws, losses = run_series(strong, weak.act, n_games=100)
print(f'Strong vs Weak -> W={wins} D={draws} L={losses}')

labels = ['Victorias\n(strong)', 'Empates', 'Derrotas\n(strong)']
plt.figure(figsize=(6, 4))
plt.bar(labels, [wins, draws, losses], color=['#4CAF50', '#FFC107', '#F44336'])
plt.title('RL-5000ep vs RL-500ep (100 partidas)')
plt.ylabel('Partidas'); plt.grid(axis='y'); plt.tight_layout(); plt.show()


## Experimento 3: QLearningAgent vs. MCTSAgentRandom (200 sims)

Reusa W_BASE — **sin reentrenar**.

In [ ]:
# Reusa W_BASE, sin reentrenar
ql_agent = make_agent(W_BASE, depth=DEPTH)

print('Jugando 50 partidas QL vs MCTS-200...')
wins, draws, losses = run_series(ql_agent, mcts200.act, n_games=50)
print(f'QL vs MCTS-200 -> W={wins} D={draws} L={losses}  ({wins*2}%)')

labels = ['QL gana', 'Empate', 'MCTS gana']
plt.figure(figsize=(6, 4))
plt.bar(labels, [wins, draws, losses], color=['#2196F3', '#FFC107', '#FF5722'])
plt.title(f'QLearningAgent ({N_EPISODES} ep, depth={DEPTH}) vs MCTSAgentRandom (200 sims)\n50 partidas')
plt.ylabel('Partidas'); plt.grid(axis='y'); plt.tight_layout(); plt.show()


## Experimento 4: Impacto de la profundidad Alpha-Beta

Todos los agentes usan **exactamente W_BASE** — solo cambia `search_depth`. Esto aísla el efecto de la busqueda online respecto al aprendizaje.

In [ ]:
# Todos usan W_BASE, solo varia search_depth
depths = [0, 2, 4, 6]
depth_wins = []

for d in depths:
    agent_d = make_agent(W_BASE, depth=d)
    w, dr, l = run_series(agent_d, mcts200.act, n_games=50)
    depth_wins.append(w)
    print(f'depth={d}: W={w} D={dr} L={l}  ({w*2}%)')

plt.figure(figsize=(7, 4))
plt.plot(depths, [w*2 for w in depth_wins], marker='s', linewidth=2, color='#2196F3')
plt.axhline(50, color='red', linestyle='--', label='50%')
plt.xlabel('Profundidad Alpha-Beta')
plt.ylabel('Win rate vs MCTS-200 (%)')
plt.title('Profundidad de busqueda vs MCTSAgentRandom (200 sims)')
plt.xticks(depths); plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()


## Conclusiones

### Por que Alpha-Beta + Q-Learning supera al lookup plano?

El agente v1 tenia **cero lookahead** y perdia contra MCTS-200.
La version mejorada combina:
- **Funcion de evaluacion aprendida** `V(s) = w * psi(s)` — captura patrones estrategicos
- **Alpha-Beta profundidad 4** — detecta forks y tacticas implicitamente

### Tabla comparativa

| Aspecto | MCTSAgentRandom | QLearning v1 | **QLearning v2** |
|---------|-----------------|--------------|------------------|
| Paradigma | Busqueda online | Aprendizaje offline | **RL + busqueda** |
| Lookahead | Si (MCTS) | No | **Si (Alpha-Beta d=4)** |
| Evaluacion | Rollouts aleatorios | Pesos aprendidos | **Pesos aprendidos** |
| Variables | sims | n_episodes | **n_episodes + depth** |
| Fundamento | Diapo 13 | Diapo 12 | **Diapo 12 + 13** |

### Observaciones

1. **`n_episodes`:** mas auto-juego mejora V(s), analogo a mas sims en MCTS (Exp 1).
2. **Truco bipolar:** propagar negando el valor cada turno `r = -gamma*r` permite aprender una sola Q-function valida para ambos jugadores (Diapo 12).
3. **Depth 0 vs depth 4:** sin busqueda el agente pierde; con depth=4 detecta tacticas como forks implicitamente (Exp 4).
4. **Limitacion:** aproximacion lineal con 8 features; DQN seria el siguiente paso.
